# VL model tesztelese DeepEval-lal


In [1]:
from src.settings.config import Config

settings = Config()

In [2]:
from src.wrapper.vl_model import QwenVLWrapper

model_to_test = QwenVLWrapper(settings.vl_model_id)
# model_to_test = QwenVLWrapper(model_id="./checkpoints/notebook-finetune")

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [3]:
from src.data.data import QwenDataset
import pandas as pd

DATASET_SPLIT = "test[90%:]"
qwen_dataset = QwenDataset(
    dataset_id=settings.dataset_id,
    split=DATASET_SPLIT,
    processor=model_to_test.processor,
    cache_dir="./data",
)

first_model_input = qwen_dataset[0]
print("Model input keys:", list(first_model_input.keys()))
print("input_ids shape:", tuple(first_model_input["input_ids"].shape))


preview_rows = []
for idx, row in enumerate(qwen_dataset.dataset):
    preview_rows.append(
        {
            "name": f"chartqa_{idx}",
            "question": row["question"],
            "expected_answer": str(row["answer"]),
        }
    )

pd.DataFrame(preview_rows).head()

Model input keys: ['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw']
input_ids shape: (981,)


,name,question,expected_answer
0,chartqa_0,What Canadian sitcom received five Golden Glob...,Schitt's Creek
1,chartqa_1,What was Lesotho's gross domestic product in 2...,2.29
2,chartqa_2,In what year did Panama's population reach 4.2...,2020
3,chartqa_3,What was the population of Panama in 2020?,4.28
4,chartqa_4,What was Cineworld Group's net income in 2019?,180.3


In [4]:
from deepeval.test_case import LLMTestCase

test_cases = []
prediction_rows = []


for idx, row in enumerate(qwen_dataset.dataset):
    prediction = model_to_test.answer(
        image=row["image"].convert("RGB"),
        question=row["question"],
        max_new_tokens=128,
    )

    test_cases.append(
        LLMTestCase(
            name=f"chartqa_{idx}",
            input=row["question"],
            actual_output=prediction,
            expected_output=str(row["answer"]),
        )
    )

    prediction_rows.append(
        {
            "name": f"chartqa_{idx}",
            "question": row["question"],
            "expected_output": str(row["answer"]),
            "actual_output": prediction,
        }
    )


predictions_df = pd.DataFrame(prediction_rows)
predictions_df

,name,question,expected_output,actual_output
0,chartqa_0,What Canadian sitcom received five Golden Glob...,Schitt's Creek,"Based on the provided bar chart, we can determ..."
1,chartqa_1,What was Lesotho's gross domestic product in 2...,2.29,"Based on the provided line chart, we can deter..."
2,chartqa_2,In what year did Panama's population reach 4.2...,2020,"Based on the bar chart provided, we can determ..."
3,chartqa_3,What was the population of Panama in 2020?,4.28,"Based on the provided bar chart, we can determ..."
4,chartqa_4,What was Cineworld Group's net income in 2019?,180.3,"Based on the provided bar chart, we can determ..."
...,...,...,...,...
245,chartqa_245,What was the number of recorded deaths from co...,167,"Based on the bar chart provided, we can determ..."
246,chartqa_246,How much did Brazil's population increase in 2...,0.75,"Based on the bar chart provided, we can determ..."
247,chartqa_247,What was the agricultural sector's contributio...,4.89,"Based on the provided bar chart, we can determ..."
248,chartqa_248,Which province had the highest relative incide...,Autonomous Province of Bolzano,"Based on the provided bar chart, we can determ..."


In [5]:
from src.wrapper.jg_model import JudgeWrapper

judge = JudgeWrapper(
    model_name=settings.jg_model_id,
    api_key=settings.groq_api_key,
    temperature=settings.judge_temperature,
)

In [6]:
from deepeval.metrics import ExactMatchMetric, AnswerRelevancyMetric
from deepeval.evaluate.configs import AsyncConfig
from deepeval import evaluate
import nest_asyncio

nest_asyncio.apply()

metrics = [
    AnswerRelevancyMetric(model=judge, threshold=0.5),
]

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(run_async=False, max_concurrent=1)
)

metric_rows = []
for test_result in results.test_results:
    if test_result.metrics_data:
        for metric_data in test_result.metrics_data:
            metric_rows.append(
                {
                    "name": test_result.name,
                    "metric": metric_data.name,
                    "score": metric_data.score,
                    "threshold": metric_data.threshold,
                    "success": metric_data.success,
                    "reason": metric_data.reason,
                }
            )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

if not metrics_df.empty:
    display(
        metrics_df.groupby("metric", as_index=False)["score"]
        .mean()
        .rename(columns={"score": "avg_score"})
    )

✨ You're running DeepEval's latest Answer Relevancy Metric! (using Groq meta-llama/llama-4-scout-17b-16e-instruct,
strict=False, async_mode=False)...

Output()



Metrics Summary

  - ❌ Answer Relevancy (score: 0.375, threshold: 0.5, strict: False, evaluation model: Groq meta-llama/llama-4-scout-17b-16e-instruct, reason: The score is 0.38 because although some relevant information might have been provided, the actual output contained several irrelevant statements that detracted from directly answering the question about the Canadian sitcom with five Golden Globe nominations in 2021, such as describing the chart's format, providing the chart's title, describing the x-axis, discussing 'The Crown', and mentioning other shows., error: None)

For test case:

  - input: What Canadian sitcom received five Golden Globe nominations in 2021?
  - actual output: Based on the provided bar chart, we can determine which Canadian sitcom received five Golden Globe nominations.

The chart is a horizontal bar graph that lists various television shows and their corresponding number of Golden Globe nominations. The title of the chart is "Number of nominations," an

⚠ WARNING: No hyperparameters logged.
» ]8;id=160795;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4092.65s | token cost: None)
» Test Results (250 total tests):
   » Pass Rate: 85.2% | Passed: 213 | Failed: 37

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

,name,metric,score,threshold,success,reason
0,chartqa_0,Answer Relevancy,0.375000,0.5,False,The score is 0.38 because although some releva...
1,chartqa_1,Answer Relevancy,1.000000,0.5,True,The score is 1.00 because the actual output di...
2,chartqa_2,Answer Relevancy,0.400000,0.5,False,The score is 0.40 because although the output ...
3,chartqa_3,Answer Relevancy,0.800000,0.5,True,The score is 0.80 because while the output lik...
4,chartqa_4,Answer Relevancy,0.400000,0.5,False,The score is 0.40 because while the output may...
...,...,...,...,...,...,...
245,chartqa_245,Answer Relevancy,0.600000,0.5,True,The score is 0.60 because while the output att...
246,chartqa_246,Answer Relevancy,1.000000,0.5,True,The score is 1.00 because the actual output di...
247,chartqa_247,Answer Relevancy,1.000000,0.5,True,The score is 1.00 because the actual output pe...
248,chartqa_248,Answer Relevancy,0.333333,0.5,False,The score is 0.33 because the actual output fa...


,metric,avg_score
0,Answer Relevancy,0.829654


In [7]:
metrics_df.to_csv("base.csv")